In [21]:
import pandas as pd

In [22]:
casehold_df = pd.read_csv("train.csv", sep=",", index_col=0).reset_index(drop=True)
casehold_df.columns = ['Citing Text', 'Holding Statement 0', 'Holding Statement 1', 'Holding Statement 2', 'Holding Statement 3', 'Holding Statement 4', 'Holding Statement 0 Similarity. w. GT Holding', 'Holding Statement 1 Similarity. w. GT Holding', 'Holding Statement 2 Similarity. w. GT Holding', 'Holding Statement 3 Similarity. w. GT Holding', 'Holding Statement 4 Similarity. w. GT Holding', 'GT Holding Label']

In [23]:
instruction_prompt = "Identify the single correct legal holding statement from options A-E to fill the <HOLDING> placeholder in the citation and output only the corresponding letter."

In [24]:
casehold_df

,Citing Text,Holding Statement 0,Holding Statement 1,Holding Statement 2,Holding Statement 3,Holding Statement 4,Holding Statement 0 Similarity. w. GT Holding,Holding Statement 1 Similarity. w. GT Holding,Holding Statement 2 Similarity. w. GT Holding,Holding Statement 3 Similarity. w. GT Holding,Holding Statement 4 Similarity. w. GT Holding,GT Holding Label
0,"Drapeau’s cohorts, the cohort would be a “vict...",holding that possession of a pipe bomb is a cr...,holding that bank robbery by force and violenc...,holding that sexual assault of a child qualifi...,holding for the purposes of 18 usc 924e that ...,holding that a court must only look to the sta...,0.000,0.652,0.647,0.670,0.639,0
1,Colameta used customer information that he too...,recognizing that even if a plaintiff claims ce...,holding that included among trade secrets empl...,holding that supplier lists can be trade secre...,recognizing that customer lists may be protect...,recognizing a legitimate need to protect an em...,0.594,0.000,0.552,0.511,0.676,1
2,property tax sale. In reviewing section 6323(b...,holding that where there is a conflict between...,holding that specific statutory provisions tak...,holding wills more specific provision prevails...,recognizing that a specific statute controls o...,holding that a specific statutory provision pr...,0.441,0.604,0.251,0.547,0.000,4
3,They also rely on Oswego Laborers’ Local 214 P...,holding that plaintiff stated a 349 claim whe...,holding that plaintiff stated a claim for brea...,holding that the plaintiff stated a claim for ...,holding that the plaintiff had not stated a cl...,holding plaintiff stated claim in his individu...,0.000,0.791,0.799,0.802,0.792,0
4,did not affect the defendant’s guideline range...,holding that united states v booker 543 us 220...,holding that waiver of right to appeal sentenc...,holding that united states v booker 543 us 220...,holding that united states v booker 543 us 220...,holding that the changes in sentencing law imp...,0.355,0.301,0.264,0.000,0.316,3
...,...,...,...,...,...,...,...,...,...,...,...,...
42504,regard. V. The appellant’s fifth argument is t...,holding that because the term conviction is am...,holding that where juror did not disclose that...,holding that a venire person who knew of the d...,holding that remand for resentencing is approp...,holding that where a juror did not disclose th...,0.744,0.423,0.755,0.733,0.000,4
42505,have been imposed” standard applies to scoresh...,holding that an error was not harmless when th...,holding that the sentencing courts expressions...,holding that where a district court clearly in...,holding that the trial court may not retain ju...,holding that when the trial court sentenced th...,0.693,0.671,0.691,0.653,0.000,4
42506,that the search was constitutional because the...,holding a fivemonth delay in searching a compu...,recognizing that a fivemonth delay weighs agai...,holding that a fivemonth delay in filing a mot...,holding the search of a computer after the war...,holding a month delay in search of a computer ...,0.000,0.630,0.603,0.647,0.590,0
42507,2001 and June 2002. None of the excerpts of th...,holding that the only fraud that could vitiate...,holding that reliance is not an element to be ...,holding that justifiable reliance is a jury qu...,holding that where a merger clause is included...,holding that a fraud class action cannot be ce...,0.533,0.687,0.617,0.000,0.658,3


In [25]:
def build_casehold_prompt(row):
    prompt_parts = [instruction_prompt, row["Citing Text"]]
    for letter, index in zip("ABCDE", range(5)):
        prompt_parts.append(f"{letter}. {row[f'Holding Statement {index}']}")
    return " ".join(prompt_parts)

def map_casehold_label(label):
    label_map = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E", "0": "A", "1": "B", "2": "C", "3": "D", "4": "E", "A": "A", "B": "B", "C": "C", "D": "D", "E": "E"}
    return label_map.get(label, label_map.get(str(label), label))

casehold_export_df = pd.DataFrame({
    "prompt": casehold_df.apply(build_casehold_prompt, axis=1),
    "gt holding label": casehold_df["GT Holding Label"].apply(map_casehold_label),
})

casehold_export_df.to_csv("casehold.tsv", sep="\t", index=False)
casehold_export_df

,prompt,gt holding label
0,Identify the single correct legal holding stat...,A
1,Identify the single correct legal holding stat...,B
2,Identify the single correct legal holding stat...,E
3,Identify the single correct legal holding stat...,A
4,Identify the single correct legal holding stat...,D
...,...,...
42504,Identify the single correct legal holding stat...,E
42505,Identify the single correct legal holding stat...,E
42506,Identify the single correct legal holding stat...,A
42507,Identify the single correct legal holding stat...,D
